# 계획

1. LLM으로 Label 생성
label 데이터 생성시에는 최대한 많은 정보를 넣어 생성.
- system prompt : 직접 작성
- user prompt : mayor_hope_xx_xx.json의 Title, Question + Answer
- assistant : 도출된 키워드 dictionary

2. 데이터셋 구성
label 생성 시 사용한 user prompt의 answer는 제외하고 나머지는 그대로 사용.


### 1. 분류할 키워드 확실히
어떤 카테고리의 분류를 해야하는지 민원 통계자료를 보며 그사람들은 어떻게 했는지 확인.

### 2. SYSTEM PROMPT 정교화.
분야 : 교통, 환경/위생, 도로/시설, 주택/건축, 복지, 보건/의료, 노동/소상공인, 공원/자연

부서 : 실제 서울시 조직도 검색해 참고.
https://org.seoul.go.kr/mobile/org/orgChart.do

너무 세분화된 부서보다 큰 실/국/본부 단위로 나누는건 어떨까? (실/국/본부 = 교통실, 기후환경본부, 민생노동국 처럼 큰 단위)
누르면 안에 뭐하는지 써져있음

긴급성 : RAGAS 설명 더 꼼꼼히
감정 상태 : RAGAS 설명 더 꼼꼼히

In [1]:
import json
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

C:\Users\User\PycharmProjects\LLM_FineTuning_1\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
SYSTEM_PROMPT = """당신은 서울시 민원 분류 담당관입니다. 지금부터 민원과 민원에 대한 답변을 읽고 키워드를 추출해주세요.
민원은 제목인 Title과 본문인 Question으로 구분되어 입력됩니다.
민원에 대한 답변은 Answer로 입력됩니다.

1. 중요도
Title과 Question을 보고 해당 민원의 중요도를 파악해 높음, 보통, 낮음 중 레이블을 구분하세요.
- 높음 : 행정적 조치, 전문적인 도움이 필요한 글. 특정한 문제가 발생했거나 부당한 처우에 대한 항의.
- 보통 : 보통의 의견이나 제안, 생각을 담은 글. 소식, 칭찬, 정보를 담은 글 등.
- 낮음 : 내용이 없이 감정적으로만 작성한 글. 도배성/어그로성 글. 특정 개인에 대한 근거 없고 맹목적인 비난 글. 비논리적이고 문맥에 일관성이 없는 글. 작성이 온전히 다 되지 않은 글.

2. 전달부서
다음은 서울시의 '교통실, 복지실, 경제실, 기후환경본부, 문화본부, 시민건강국, 재난안전실, 주택실, 여성가족실'과 각 부서가 담당하는 분야입니다. 민원 내용을 보고 해당 민원이 전달되어야 할 부서를 골라주세요.

- 교통실 : 버스·지하철·택시, 대중교통 정책, 자전거·킥보드·보행, 주차, 신호, 불법주정차, 한강버스, 교통카드, 도로교통, 자율주행
- 복지실 : 기초생활보장·긴급복지, 저소득층 지원, 노숙인, 어르신 돌봄·요양, 장애인 지원, 아동·청소년 복지, 한부모·다문화가족, 사회적 고립, 중장년 지원, 공익활동
- 경제실 : 창업·스타트업, 소상공인·전통시장 지원, 청년 취업·일자리, 중소기업 자금, 소비자 권익, 생활임금·노동정책, 직장 내 괴롭힘, 사회적경제, 도시농업, 국제협력
- 기후환경본부 : 쓰레기·재활용, 소각장, 미세먼지·대기질, 소음·악취, 동물보호·유기동물, 탄소중립·신재생에너지, 친환경차·전기차 충전, 도시공원, 식품안전, 생활환경(석면·화학물질)
- 문화본부 : 도서관, 박물관·문화시설, 공연·예술 지원, 문화유산·한양도성, 공공디자인, 옥외광고물, 야간경관·빛축제, 전통문화, 과학관, 미디어아트
- 시민건강국 : 보건소, 응급의료, 감염병·방역, 정신건강·위기상담, 예방접종, 치매 예방, 공중위생, 건강증진, 마약 대응, 산후조리·영유아 건강
- 재난안전실 : 재난대응·안전상황실, 한파·폭염·풍수해, 지반침하·취약시설 점검, 도로·보도 안전, 야간 안전, 민방위·대피소, 시민안전보험, 제설, 인파 안전관리, 중대재해
- 주택실 : 재개발·재건축, 공공주택·청년안심주택, 전세사기·임차보증금, 건축허가·인허가, 도시계획, 빈집 정비, 공동주택 관리, 주거환경개선, 도시재생, 부동산 정보
- 여성가족실 : 보육·어린이집, 아이돌봄 서비스, 저출생 대응, 아동학대 예방, 청소년 지원·보호, 성폭력·성희롱 예방, 디지털성범죄, 여성 안전, 키움센터, 양성평등

단, Answer를 제외한 민원(Title과 Question)을 보았을 때 다음의 경우에 해당한다면 '분류 보류'를 설정하세요.
- Title과 Question만으로 민원의 주제를 알 수 없어 특정 부서를 분류할 수 없는 경우
- 첨부 파일을 업로드 했다고 되어 있으나 Title과 Question만으로 어떤 내용인지 유추할 수 없는 경우.
- Title과 Question이 내용을 알 수 없을 정도로 짧은 경우.

3. 민원 유형
Title과 Question을 보고 민원의 유형을 다음 중 하나로 구분하세요
- 신고 : 불법 행위, 위험 상황, 규정 위반 등 제3자나 시설에 대한 문제를 알리는 경우
- 문의 : 제도, 정책, 절차, 방법 등에 대한 정보나 안내를 요청하는 경우
- 건의 : 정책 개선, 시설 설치, 제도 변경 등을 제안하는 경우
- 항의 : 행정 처리나 처우에 대한 불만을 표출하거나 시정을 요구하는 경우
- 칭찬 : 공무원, 서비스, 정책 등에 대한 긍정적인 평가를 담은 경우.
- 그 외 : 위 유형 중 어느 것으로도 분류되지 않는 경우.

4. 감성상태
Title과 Question을 보고 민원인의 감정상태를 긍정, 중립, 부정 중 하나로 구분하세요.

{format_instructions}
"""

In [15]:
class MinionLabel(BaseModel):
    중요도: str = Field(description="민원의 중요도. 높음/보통/낮음 중 하나")
    담당부서: str = Field(description="민원이 전달되어야 하는 서울시 부서. 여성가족실/주택실/재난안전실/시민건강국/문화본부/기후환경본부/경제실/복지실/교통실/분류 보류 중 하나.")
    민원유형: str = Field(description="민원의 유형. 신고/문의/건의/칭찬/항의/그 외 중 하나")
    감정상태: str = Field(description="민원인의 감정 상태. 긍정/중립/부정 중 하나")

parser = JsonOutputParser(pydantic_object=MinionLabel)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", """Title: {title}
Question: {question}
Answer: {answer}""")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# Anthropic
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)

# OpenAI
#llm = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)

chain = prompt | llm | parser

# 실행코드
100건 실행시마다 csv에 그대로 저장

In [5]:
def run_chain(df: pd.DataFrame, output_path: str):
    #초기설정 : assistant 열 만들기, 변수 설정
    if "assistant" not in df.columns:
        df["assistant"] = None

    total = len(df)

    for i, row in df.iterrows():
        #assistant 열이 없는 경우만 실행
        if pd.notna(row["assistant"]):
            continue

        #LLM 실행 후 저장
        try:
            result = chain.invoke({
                "title": row["title"],
                "question": row["Question"],
                "answer": row["Answer"]
            })
            df.at[i, "assistant"] = json.dumps(result, ensure_ascii=False)
            print(f"[{i+1}/{total}] 완료: {result}")

        except Exception as e:
            print(f"[{i+1}/{total}] 스킵: {e}")
            df.at[i, "assistant"] = "ERROR"

        #100건마다 저장
        if (i + 1) % 50 == 0:
            df.to_csv(output_path, index=False)
            print(f"--- {i+1}건 중간 저장 완료 ---")

    df.to_csv(output_path, index=False)
    print(f"\n완료! 총 {total}건 처리")

# 모델별 테스트 진행

| 모델명                      | 평가 | 50건 처리 가격 (달러) | 비고                         |
  |--------------------------|---|---------------|----------------------------|
  | gpt-4o-mini              | 평가 전 | <0.01 (10원 미만) | 비용 최저                      |
  | gpt-4o                   | 평가 전 | 0.27 (400원)   | TPM 자주 걸림                  |
  | claude-sonnet-4-20250514 | 평가 전 | 0.6 (890원)    | 레거시 모델이라 같은 가격이면 4.6이 더 나음 |
  | claude-sonnet-4-6        | 평가 전 | 0.6 (890원)    | 5분 넘게 걸림                   |
  | claude-haiku-4-5-20251001| 평가 전 | 음음            | 없음


In [17]:
# 테스트셋 생성

df = pd.read_csv("mayor_hope_labeled.csv")
df_test = df.sample(n=50, random_state = 99)

run_chain(df_test, "test_claude-haiku-4-5-20251001.csv") #

[2617/50] 완료: {'중요도': '보통', '담당부서': '교통실', '민원유형': '건의', '감정상태': '부정'}
[5639/50] 완료: {'중요도': '보통', '담당부서': '문화본부', '민원유형': '건의', '감정상태': '긍정'}
[3239/50] 완료: {'중요도': '보통', '담당부서': '문화본부', '민원유형': '항의', '감정상태': '부정'}
[6541/50] 완료: {'중요도': '높음', '담당부서': '교통실', '민원유형': '항의', '감정상태': '부정'}
[11063/50] 완료: {'중요도': '보통', '담당부서': '기후환경본부', '민원유형': '건의', '감정상태': '부정'}
[11290/50] 완료: {'중요도': '보통', '담당부서': '교통실', '민원유형': '건의', '감정상태': '부정'}
[4263/50] 완료: {'중요도': '높음', '담당부서': '재난안전실', '민원유형': '신고', '감정상태': '부정'}
[7400/50] 완료: {'중요도': '높음', '담당부서': '주택실', '민원유형': '항의', '감정상태': '부정'}
--- 7400건 중간 저장 완료 ---
[3887/50] 완료: {'중요도': '낮음', '담당부서': '분류 보류', '민원유형': '항의', '감정상태': '부정'}
[1702/50] 완료: {'중요도': '낮음', '담당부서': '분류 보류', '민원유형': '항의', '감정상태': '부정'}
[131/50] 완료: {'중요도': '보통', '담당부서': '여성가족실', '민원유형': '건의', '감정상태': '부정'}
[1836/50] 완료: {'중요도': '보통', '담당부서': '교통실', '민원유형': '건의', '감정상태': '부정'}
[2474/50] 완료: {'중요도': '보통', '담당부서': '교통실', '민원유형': '건의', '감정상태': '중립'}
[4240/50] 완료: {'중요도': '높음', '담당부서': '주택실

In [37]:
df_print = pd.read_csv("t.csv")

for _, row in df_print.iterrows():
    print(f"[title]\n{row['title']}")
    print(f"[Question]\n{row['Question']}")
    print(f"[assistant]\n{row['assistant']}")
    print("=" * 40)

[title]
서울의 공공디자인은 문화컨텐츠 스토리텔링으로 리모델링을----
[Question]
시장님! 
서울의 디자인서울정책은 공공디자인 정책으로 많은 변화를 가져왔습니다. 물론 부정적 이미지와 긍정적 이미지의 두가지 축에서 생각을 해봅니다만, 새롭게 시작하는 축 보다는 리모델링 기법의 정책적용, 시정이 필요한 싯점입니다.
(보여지는 각25개 지자체의 공공디자인거리 사업으로 토목공사 위주로 모양만 갖추어 놓은 상황과 한강의 여러가지 프로젝트들이 있는데, 이제는 알토란 같은 디자인 스토리텔링의 적용으로 문화적 역량을 가미 하여 살을 붙이고 결실을 맺여야 합니다.)서울을 유네스코에 문화유산으로 등재하여 관광 상품화하는데 등재기여 했고, 여러부문에 영향을 끼쳐 다변화하는 문화콘텐츠로서의 서울을 만들어 낼려고 단초를 한것 같습니다. 이제 .외형적인 디자인 서울 시대를 거쳐 이제는 실속있는 각개분야의 역사와 흥미 재미가 있는 서울을 만들어 내어 서울시민과 서울을 방문하는 외지인 (외국인 포함)에게 스토리 텔링의 문화 컨텐츠를 제공하는 디자인서울을 만들어가야 하지 않을까 생각합니다. 물론 형태적 공간적인 모습들의 마무리는 지어가면서 즐겁고 흥겨운 문화시민으로서 서울을 긍지 있고 자랑스럽게 만들어 가야 하지 않을까요? 서울문화디자인사랑 서포터츠를 만들고 동대문 디자인플라자의 문화 컨텐츠 서울의 명품 문화상품과 한류K시리츠 문화컨텐츠인 문화정책으로 연계하고 컨텐츠를 개발하여 어울리고 스토리와 감동이 있는 문화콘텐츠 스토리텔링관광으로 더욱 활력있고 명분있는 온,오프라인콘텐츠 축과 더불어 다양한 즐길꺼리 볼꺼리를 만들어 역사와 전통의 서울 문화콘텐츠로 생산성있는 아이러브 서울을 기대해 봅니다. 기회가 되면 참여하고 싶구요.
[assistant]
{"감정상태": "긍정", "중요도": "보통", "담당부서": "문화본부"}
[title]
지하철 안내방송과 모니터에 관한 것
[Question]
박원순 시장님 안녕하셨습니까.
오늘이 금년 마지막이고, 새해를 맞이하게 되었습니다.


In [ ]:
# 모델별 출력 평가
모델에 들어가는 prompt 토큰수는 많아봐야 2k. LLM의 컨텍스트가 모자라지는 않음. 다만, 긴 system prompt를 보면서 중간 소실 없이 잘 분류했는지 판단하기 위해서 내가 직접 정답을 만들어 평가하는게 좋을 듯.

1. gpt-4o TPM 때문에 비어있는거 채우기
2. gemini api 실험 추가